In [ ]:
%load_ext autoreload
%autoreload 2
    
%matplotlib widget

from pySEA.sea_eco.io import collect_swift_file
from pySEA.sea_eco.io import swift_to_sea_metadata #Metadata, 
from pySEA.sea_eco.architecture.base_structure_numpy import Metadata

import numpy as np
from pprint import pprint

# Folder

In [ ]:
base = '../example_data/'

In [3]:
files = ['Snapshots - _Ronchi-Orca-1-Display-Snapshot', #snapshot
         'Aquire - _Acquire-HAADF-2',
         'Aquire - _Acquire-Ronchigram-1', #raw data
         'Integrated - Sequence Ronchigram  - 20241125 hBN', #processed
         'Sequence-Synchronized-HAADF-7',
         'Sequence-Synchronized-MAADFBF-7',
         'Sequence - Sequence Synchronized EELS 7',
         #'Sequence  - Sequence Ronchigram  - 20241125 hBN',
         'Tableau dectris_ela 8'
         ]

files= files

# Test collect file

In [4]:
for f in files:
    meta, data = collect_swift_file(base+f)
    print(f'File name: {f}')
    print(f'Shape:     {data.shape}')
    pprint(meta)
    print()

File name: Snapshots - _Ronchi-Orca-1-Display-Snapshot
Shape:     (512, 512)
{'collection_dimension_count': 0,
 'datetime_modified': {'dst': '+60',
                       'local_datetime': '2025-06-17T11:13:01.711486',
                       'timezone': 'America/New_York',
                       'tz': '-0400'},
 'datetime_original': {'dst': '+60',
                       'local_datetime': '2025-06-17T11:13:01.711486',
                       'timezone': 'America/New_York',
                       'tz': '-0400'},
 'datum_dimension_count': 2,
 'intensity_calibration': {'offset': 0.0, 'scale': 1.0, 'units': 'counts'},
 'large_format': False,
 'metadata': {'hardware_source': {'autostem': {'Acquisition:Binning': 4.0,
                                               'Acquisition:DarkMode': 'linear',
                                               'Acquisition:ExposureTime(s)': 0.200003,
                                               'Acquisition:GainMode': 0.0,
                                    

# Test metadata class
Check that metadata can be read from all of the test files and converted to the GeneralMetada class

In [ ]:
# try:
for i, f in enumerate(files):
    print(f'{i}: {f}')
    meta, data = collect_swift_file(base+f)
    print(f'  shape: {data.shape}')
    og_meta = Metadata(meta)
    print(f'  #dimensions: {len(og_meta.spatial_calibrations)}')
    sea_meta = swift_to_sea_metadata(og_meta,
                                        #print_missed=True,
                                        file_path=base+files[i],
                                        read_aberrations=True)
    print()
print('Successfully read metadata from all files.')
# except:
#     raise RuntimeError(f'Failed to read metadata from {f}. Check the file path and format.')

0: Snapshots - _Ronchi-Orca-1-Display-Snapshot
  shape: (512, 512)
  #dimensions: 2

1: Aquire - _Acquire-HAADF-2
  shape: (64, 32)
  #dimensions: 2

2: Aquire - _Acquire-Ronchigram-1
  shape: (512, 512)
  #dimensions: 2

3: Integrated - Sequence Ronchigram  - 20241125 hBN
  shape: (2048, 2048)
  #dimensions: 2

4: Sequence-Synchronized-HAADF-7
  shape: (2, 64, 32)
  #dimensions: 3

5: Sequence-Synchronized-MAADFBF-7
  shape: (2, 64, 32)
  #dimensions: 3

6: Sequence - Sequence Synchronized EELS 7
  shape: (2, 64, 32, 32)
  #dimensions: 4

7: Tableau dectris_ela 8
  shape: (10, 5, 16, 32)
  #dimensions: 4

Successfully read metadata from all files.


Aberrations not found for
- Any Ronchi datasets
- Tableau dectris_ela 8 (i=6)

They do not appear in the metadata either

## Test: Swift to GeneralMetada
Test that swift metadata is correctly converted from the collected json to metadata

In [ ]:
i = 6
print(files[i])
original = Metadata(collect_swift_file(base+files[i])[0])
original.show_tree()

Sequence - Sequence Synchronized EELS 7


## Test: Swift metadata to SEA Metada
Test that swift metadata is correctly converted from the collected json to SEA metadata

In [ ]:
i = 6
print(files[i])
meta, data = collect_swift_file(base+files[i])
original = Metadata(meta)
new1 = swift_to_sea_metadata(original, print_missed=True, file_path=base+files[i])
new1.show_tree()

Sequence - Sequence Synchronized EELS 7


# Dimensions from SEA metadata

The following dimensions test will test dimensions created from metadata, so we need some initial metadata.<br />
Get metadata from swift and convert to SEA metadata

In [ ]:
from pySEA.sea_eco.io import infer_dimensions

i = 6
print(files[i])
meta, data = collect_swift_file(base+files[i])
original = Metadata(meta)
new = swift_to_sea_metadata(original, print_missed=True, file_path=base+files[i])
print(infer_dimensions(new.Dimensions, data.shape))
new.show_tree()

Sequence - Sequence Synchronized EELS 7
1D-EELS


# Load swift signal to SEA signal
Loading of swift file directly to a signal class

In [ ]:
from pySEA.sea_eco.io import load_swift_to_sea

In [10]:
#i = 5
i = 6
signal = load_swift_to_sea(base+files[i])
signal.show_tree()

# Test SignalSet


In [ ]:
from pySEA.sea_eco.architecture.base_structure_numpy import SignalSet, AcquisitionSet

display('SignalSet')
SignalSet().show_tree()
display('AquisitionSet')
AcquisitionSet().show_tree()

'SignalSet'

'AquisitionSet'

In [12]:
signals = [4,5,6]
names = ['HAADF', 'MAADF', 'SI']
signals = [load_swift_to_sea(base+files[i]) for i in signals]
for signal,name in zip(signals,names): signal.name=name
print('Signals in set:')
pprint(signals)

signal_set = SignalSet(signals=signals, main_signal=-1)
signal_set.show_tree()

Signals in set:
[<Signal name="HAADF" signal_type=None dimensions_domain=local>,
 <Signal name="MAADF" signal_type=None dimensions_domain=local>,
 <Signal name="SI" signal_type=1D-EELS dimensions_domain=local>]


In [13]:
signals = [4,5,6]
names = ['HAADF', 'MAADF', 'SI']
signals = [load_swift_to_sea(base+files[i]) for i in signals]
for signal,name in zip(signals,names): signal.name=name
print('Signals in set:')
pprint(signals)

acquisition_set = AcquisitionSet(signals=signals, main_signal=-1)
acquisition_set.show_tree()

Signals in set:
[<Signal name="HAADF" signal_type=None dimensions_domain=local>,
 <Signal name="MAADF" signal_type=None dimensions_domain=local>,
 <Signal name="SI" signal_type=1D-EELS dimensions_domain=local>]
